In [ ]:
import pandas as pd
import numpy as np

df_proj= pd.read_csv(r'D:\AnacondaProj\3_days_Project_BusinessLAB\data\users\LabData\raw\tblprojects.csv')

df_task = pd.read_csv(r'D:\AnacondaProj\3_days_Project_BusinessLAB\data\users\LabData\raw\tbltasks.csv')
df_task .head(5)

In [ ]:
df_proj.info()

# Converting Data Type

start_date -> dat_time>br>
deadline -> date_time<br>
project_created -> date_time<br>
date_finished -> Date_time<br>

Converting This columns Data Type to Date Time


In [ ]:
import pandas as pd

df_proj["start_date"] = pd.to_datetime(df_proj["start_date"])
df_proj["deadline"] = pd.to_datetime(df_proj["deadline"])
df_proj["project_created"] = pd.to_datetime(df_proj["project_created"])
df_proj["date_finished"] = pd.to_datetime(df_proj["date_finished"])

In [ ]:
df_proj.columns

# Checking Null Values in %

In [ ]:
df_proj.isnull().sum()/len(df_proj)*100

**This are Null Values in Percentage(%)**

# Cleaning Description Columns

In [ ]:
df_proj[['description']].sample(5)

In [ ]:
df_proj.loc[386, "description"]

In [ ]:
from bs4 import BeautifulSoup
import re

def clean_html(text):
    if pd.isna(text):
        return ""
    soup = BeautifulSoup(text, "html.parser")
    clean_text = soup.get_text(separator=" ")
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()
    return clean_text

df_proj["description"] = df_proj["description"].apply(clean_html)
df_proj["description"]


In [ ]:
print("\n".join(df_proj.loc[386, "description"].split(". ")))

# Cleaning Name Columns

In [ ]:
df_proj[['name']]
print("\n".join(df_proj.loc[388, "name"].split(". ")))

In [ ]:
df_proj[df_proj["name"].str.contains("api", case=False, na=False)][["name"]]

**We Will Keep name columns same coz it contain name of proj and some discription process**

In [ ]:
df_proj[df_proj["description"].str.contains("api", case=False, na=False)][["description"]]
df_proj.loc[46, "description"]


In [ ]:

for desc in df_proj[df_proj["description"].str.contains("api", case=False, na=False)]["description"]:
    print("\n", desc)

In [ ]:
import re

# Cheking is there any api keys 
pattern = r'(api[_-]?key\s*[:=]\s*[A-Za-z0-9\-_]{16,})'

matches = []

for desc in df_proj["description"].dropna():
    found = re.findall(pattern, desc, flags=re.IGNORECASE)
    matches.extend(found)

matches

**There is no real api key stores so its fine**

**Table Files Completed**

# Task File Understanding

In [ ]:
df_task.head(1)

In [ ]:
df_task_info = df_task.dtypes.reset_index()
df_task_info.columns = ["column", "dtype"]

df_task_info["non_null_count"] = df_task.count().values
df_task_info["null_count"] = df_task.isnull().sum().values

# sample values (first non-null value from each column)
df_task_info["sample_value"] = [
    df_task[col].dropna().iloc[0] if not df_task[col].dropna().empty else None
    for col in df_task.columns
]

df_task_info

# Change Data Types

In [ ]:
# numeric columns
num_cols = [
    "id", "priority", "addedfrom", "is_added_from_contact", "status",
    "repeat_every", "recurring", "is_recurring_from", "cycles",
    "total_cycles", "custom_recurring", "rel_id", "is_public",
    "billable", "billed", "invoice_id", "hourly_rate",
    "milestone", "kanban_order", "milestone_order",
    "visible_to_client", "deadline_notified"
]

df_task[num_cols] = df_task[num_cols].apply(pd.to_numeric, errors="coerce")

# datetime columns
date_cols = ["dateadded", "startdate", "duedate", "datefinished", "last_recurring_date"]

df_task[date_cols] = df_task[date_cols].apply(
    pd.to_datetime, errors="coerce", dayfirst=True
)

# categorical / text (keep as string)
cat_cols = ["name", "description", "recurring_type", "rel_type"]

df_task[cat_cols] = df_task[cat_cols].astype("string")

In [ ]:
df_task[df_task.duplicated()]

**We Will keep this as it is According to business problem there is no in need to fill nan values**

In [ ]:
df_task.columns

In [ ]:
df_task['milestone_order'].unique()

# Clean Describtion Columns

In [ ]:
df_task[['description']]

In [ ]:
def clean_html(text):
    if pd.isna(text):
        return ""
    soup = BeautifulSoup(text, "html.parser")
    clean_text = soup.get_text(separator=" ")
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()
    return clean_text

df_task["description"] = df_task["description"].apply(clean_html)
df_task["description"]

# Understand PDF

In [ ]:
from pdfminer.high_level import extract_text

text = extract_text(r"D:\AnacondaProj\3_days_Project_BusinessLAB\data\users\LabData\raw\tbltickets.pdf")
text.split("\n\n")

In [ ]:
# remove page numbers
import re
text = str(text)
text = re.sub(r'Page number:.*', '',text)
print(text)

In [ ]:
 # remove database headers / junk
text = re.sub(r'Database:.*', '',text)
# remove extra symbols
text = re.sub(r'[^\w\s@.:/-]', ' ', text)
# remove multiple spaces/newlines
text = re.sub(r'\s+', ' ', text)

text = text.strip()
print(text[:200])

In [ ]:
njn

In [ ]:
chunks = re.split(r'\n{2,}|\.\s', text)
# remove small junk chunks
chunks = [c.strip() for c in chunks if len(c) > 50]

len(chunks)
print((chunks[:300]))

In [ ]:
keywords = ["ticket", "email", "issue", "client", "task", "project"]

filtered_chunks = [
    c for c in chunks if any(k in c.lower() for k in keywords)
]
len(filtered_chunks)

In [ ]:
print((filtered_chunks[:300]))

In [ ]:
import re

def clean_chunk(chunk):
    chunk = str(chunk)

    # remove hashes / random ids
    chunk = re.sub(r'\b[a-f0-9]{6,}\b', '', chunk)

    # remove repeated words
    chunk = re.sub(r'\b(\w+)( \1\b)+', r'\1', chunk)

    # remove single letters / broken tokens
    chunk = re.sub(r'\b[a-zA-Z]\b', '', chunk)

    # remove numbers-only junk
    chunk = re.sub(r'\b\d+\b', '', chunk)

    # remove technical headers
    junk_words = [
        "ticketid", "adminr", "userid", "contactid", "merged",
        "_ticket_id", "depart", "priority", "status",
        "service", "ticketkey", "project_id", "clientread",
        "adminread", "assigned", "staff_id", "replying"
    ]

    for word in junk_words:
        chunk = re.sub(word, '', chunk, flags=re.IGNORECASE)

    # remove extra spaces
    chunk = re.sub(r'\s+', ' ', chunk)

    return chunk.strip()

In [ ]:
cleaned_chunks = [clean_chunk(c) for c in chunks]

# keep only meaningful sentences
final_chunks = [
    c for c in cleaned_chunks 
    if len(c) > 40 and any(x in c.lower() for x in [
        "please", "issue", "email", "request", "add", "confirm", "sent"
    ])
]
final_chunks = list(set(final_chunks))

final_chunks[:5]

In [ ]:
import re

def extract_meaningful_text(chunks):
    clean_data = []

    for c in chunks:
        c = str(c)

        # remove urls
        c = re.sub(r'http\S+', '', c)

        # remove html/junk words
        c = re.sub(r'\b(br|href|html|target|blank|NULL)\b', '', c, flags=re.IGNORECASE)

        # remove emails formatting noise
        c = re.sub(r'lt|gt|/|_', ' ', c)

        # remove random ids / hashes
        c = re.sub(r'\b[a-f0-9]{6,}\b', '', c)

        # remove repeated words
        c = re.sub(r'\b(\w+)( \1\b)+', r'\1', c)

        # remove extra spaces
        c = re.sub(r'\s+', ' ', c).strip()

        # keep only meaningful sentences
        if len(c) > 40 and any(word in c.lower() for word in [
            "please", "issue", "sorry", "request", "help", "need", "confirm", "delay"
        ]):
            clean_data.append(c)

    return list(set(clean_data))  # remove duplicates

In [ ]:
final_emails = extract_meaningful_text(chunks)

for i in final_emails[:5]:
    print("\n", i)